## Your first Frontier LLM Project

Let's build a useful LLM solution - in a matter of minutes.

By the end of this course, you will have built an autonomous Agentic AI solution with 7 agents that collaborate to solve a business problem. All in good time! We will start with something smaller...

Our goal is to code a new kind of Web Browser. Give it a URL, and it will respond with a summary. The Reader's Digest of the internet!!

## If you're new to Jupyter Lab

Welcome to the wonderful world of Data Science experimentation! Once you've used Jupyter Lab, you'll wonder how you ever lived without it. Simply click in each "cell" with code in it, such as the cell immediately below this text, and hit Shift+Return to execute that cell. As you wish, you can add a cell with the + button in the toolbar, and print values of variables, or try out variations.


In [23]:
# imports

import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

# Connecting to Ollama

# And now the change for Ollama

1. No environment variables are needed (no keys) so this part has been removed

2. The OpenAI client library is being initialized to point to your local computer for Ollama

3. You need to have installed Ollama on your computer, and run `ollama run llama3.2` in a Powershell or Terminal if you haven't already

4. Anywhere in this lab that it used to have **gpt-4o-mini** it now has **llama3.2**


In [24]:
# Here it is - see the base_url

openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


## 🔑 Want to use OpenAI instead of Ollama?

**This is the only cell you need to change.** All the code below (Website class, summarize, display_summary) works exactly the same.

To switch to OpenAI:
1. Get an API key from [platform.openai.com](https://platform.openai.com) → API Keys
2. Create a `.env` file in this folder with: `OPENAI_API_KEY=sk-proj-your-key-here`
3. Comment out the Ollama cell above and uncomment the cell below

In [ ]:
# --- OPENAI ALTERNATIVE ---
# Comment out the Ollama cell above and uncomment this to use OpenAI instead.
# Everything else in the notebook stays exactly the same.

# import os
# from dotenv import load_dotenv

# load_dotenv(override=True)
# api_key = os.getenv('OPENAI_API_KEY')

# if not api_key:
#     print("No API key found. Add OPENAI_API_KEY=sk-proj-... to your .env file.")
# elif not api_key.startswith('sk-proj-'):
#     print("Key found but doesn't start with sk-proj- — double check your key.")
# else:
#     print("OpenAI API key loaded!")

# openai = OpenAI(api_key=api_key)

# NOTE: Also change model='llama3.2' to model='gpt-4.1-mini' in the summarize() function below.

# Let's make a quick call to a Frontier model to get started, as a preview!

In [25]:
# To give you a preview -- calling OpenAI with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

message = "Hello, Llama! This is my first ever message to you! Hi!"
response = openai.chat.completions.create(model="llama3.2", messages=[{"role":"user", "content":message}])
print(response.choices[0].message.content)

Hi there! Welcome to our conversation! I'm happy to meet you and help with any questions or topics you'd like to discuss. Don't worry if this is your first message - I'm here to assist and learn alongside you. How's your day going so far?


## OK onwards with our first project

In [26]:
# A class to represent a Webpage
# If you're not familiar with Classes, check out the "Intermediate Python" notebook

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [27]:
# Let's try one out. Change the website and add print statements to follow along.

wiki = Website("https://en.wikipedia.org/wiki/Python_(programming_language)")
print(wiki.title)
print(wiki.text)

Python (programming language) - Wikipedia
Jump to content
Main menu
Main menu
move to sidebar
hide
Navigation
Main page
Contents
Current events
Random article
About Wikipedia
Contact us
Contribute
Help
Learn to edit
Community portal
Recent changes
Upload file
Special pages
Search
Search
Appearance
Donate
Create account
Log in
Personal tools
Donate
Create account
Log in
Contents
move to sidebar
hide
(Top)
1
History
2
Design philosophy and features
3
Syntax and semantics
Toggle Syntax and semantics subsection
3.1
Indentation
3.2
Statements and control flow
3.3
Expressions
3.4
Typing
3.5
Arithmetic operations
3.6
Function syntax
4
Code examples
5
Libraries
6
Development environments
7
Implementations
Toggle Implementations subsection
7.1
Reference implementation
7.2
Limitations of the reference implementation
7.3
Other implementations
7.4
Unsupported implementations
7.5
Transpilers to other languages
7.6
Performance
8
Language development
9
Naming
10
Languages influenced by Python
11
See 

## Types of prompts

You may know this already - but if not, you will get very familiar with it!

Models like GPT4o have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [28]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

In [29]:
# A function that writes a User Prompt that asks for summaries of websites:

def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

In [30]:
print(user_prompt_for(wiki))

You are looking at a website titled Python (programming language) - Wikipedia
The contents of this website is as follows; please provide a short summary of this website in markdown. If it includes news or announcements, then summarize these too.

Jump to content
Main menu
Main menu
move to sidebar
hide
Navigation
Main page
Contents
Current events
Random article
About Wikipedia
Contact us
Contribute
Help
Learn to edit
Community portal
Recent changes
Upload file
Special pages
Search
Search
Appearance
Donate
Create account
Log in
Personal tools
Donate
Create account
Log in
Contents
move to sidebar
hide
(Top)
1
History
2
Design philosophy and features
3
Syntax and semantics
Toggle Syntax and semantics subsection
3.1
Indentation
3.2
Statements and control flow
3.3
Expressions
3.4
Typing
3.5
Arithmetic operations
3.6
Function syntax
4
Code examples
5
Libraries
6
Development environments
7
Implementations
Toggle Implementations subsection
7.1
Reference implementation
7.2
Limitations of the re

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]

To give you a preview, the next 2 cells make a rather simple call - we won't stretch the mighty GPT (yet!)

In [31]:
messages = [
    {"role": "system", "content": "You are a funny assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

In [32]:
# To give you a preview -- calling OpenAI with system and user messages:

response = openai.chat.completions.create(model="llama3.2", messages=messages)
print(response.choices[0].message.content)

The answer to the ultimate question of life, the universe, and everything... is... *dramatic pause*... 4!

(Sorry, I couldn't resist throwing in a bit of Douglas Adams' Douglas there)


## And now let's build useful messages for GPT-4o-mini, using a function

In [33]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

In [34]:
# Try this out, and then try for a few more websites

messages_for(wiki)

[{'role': 'system',
  'content': 'You are an assistant that analyzes the contents of a website and provides a short summary, ignoring text that might be navigation related. Respond in markdown.'},
 {'role': 'user',
  'content': 'You are looking at a website titled Python (programming language) - Wikipedia\nThe contents of this website is as follows; please provide a short summary of this website in markdown. If it includes news or announcements, then summarize these too.\n\nJump to content\nMain menu\nMain menu\nmove to sidebar\nhide\nNavigation\nMain page\nContents\nCurrent events\nRandom article\nAbout Wikipedia\nContact us\nContribute\nHelp\nLearn to edit\nCommunity portal\nRecent changes\nUpload file\nSpecial pages\nSearch\nSearch\nAppearance\nDonate\nCreate account\nLog in\nPersonal tools\nDonate\nCreate account\nLog in\nContents\nmove to sidebar\nhide\n(Top)\n1\nHistory\n2\nDesign philosophy and features\n3\nSyntax and semantics\nToggle Syntax and semantics subsection\n3.1\nInden

## Time to bring it together - the API for OpenAI is very simple!

In [35]:
# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model = "llama3.2",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [36]:
summarize("https://en.wikipedia.org/wiki/Python_(programming_language)")

'Here\'s a summary of the article "Python" on Wikipedia:\n\n**Introduction**\n\nPython is a high-level, interpreted programming language that is widely used for various purposes such as web development, scientific computing, data analysis, artificial intelligence, and more.\n\n**History**\n\nPython was created in 1991 by Guido van Rossum. It is named after the British comedy group Monty Python\'s Flying Circus, which Van Rossum admired.\n\n**Implementations**\n\nThere are several implementations of Python, including:\n\n* **CPython**: The original implementation of Python\n* **IronPython**: A variant of Python that runs on .NET Common Language Runtime (CLR)\n* **PyPy**: An alternative implementation of Python that uses Just-In-Time (JIT) compilation\n\n**Features**\n\nPython is known for its simplicity, readability, and ease of use. It has a large standard library and a vast number of third-party libraries available.\n\n**Advantages**\n\nSome of the advantages of using Python include:\

In [37]:
# A function to display this nicely in the Jupyter output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [38]:
display_summary("https://en.wikipedia.org/wiki/Python_(programming_language)")

A vast and complex topic indeed!

Here's a concise summary:

**What is Python?**

Python is a high-level, interpreted programming language with a simple syntax and that is relatively easy to learn. It was created in the late 1980s by Guido van Rossum and first released in 1991.

**Key Features:**

* Highly readable code
* Large standard library with many modules and libraries for various tasks
* Extensible through third-party libraries and frameworks
* Dynamic typing, making it versatile and flexible

**Uses:**

Python is a general-purpose language, widely used in:

* Web development (e.g., Django, Flask)
* Data science and machine learning (e.g., NumPy, pandas, scikit-learn)
* Scripting (e.g., automation, data processing)
* Scientific computing (e.g., scientific libraries like SciPy)
* Education and research
* Networking and cybersecurity

**Popularity:**

Python is the language of choice for many organizations, researchers, and developers due to its:

* Relativism ease in learning
* Flexible syntax
* Large community support (over 2 million users worldwide)

**Resources:**

The official Python website provides extensive documentation, tutorials, and guides for new users. Additionally, various frameworks, libraries, and tools are available, such as:

* Python.org
* PyCharm IDE
* Jupyter Notebook
* Anaconda Distribution

Some notable variants of Python include:

* CPython (official implementation)
* IronPython (for .NET development)
* MicroPython (for embedded systems)
* NumPy-inspired libraries like SciPy and Pandas

# Let's try more websites

Note that this will only work on websites that can be scraped using this simplistic approach.

Websites that are rendered with Javascript, like React apps, won't show up. See the community-contributions folder for a Selenium implementation that gets around this. You'll need to read up on installing Selenium (ask ChatGPT!)

Also Websites protected with CloudFront (and similar) may give 403 errors - many thanks Andy J for pointing this out.

But many websites will work just fine!

In [21]:
display_summary("https://cnn.com")

**Website Summary**

### News and Announcements

- US aid cuts
- Maduro raid bet
- China's AI upstart
- Motorcycle megastar
- Mars discovery
- Milan Design Week
- Real-life Narnia
- World Cup final tickets
- Trump’s inauspicious defense of a soldier accused of insider trading on Polymarket

- Jerome Powell arrested for betting on Maduro capture
- US freezes $344 million in cryptocurrency said to be linked to Iran
- Analysis: Asia's spiraling supply shock is coming for America
- Video: 'The time for free-riding is over': Hegseth urges allies to take larger role in Strait of Hormuz

### CNN Exclusive

- Zamil Limon, missing University of South Florida doctoral student found dead and roommate named as suspect
- Swalwell accuser cooperating with Manhattan DA investigation
- Trump’s Justice Department is bringing back firing squads for federal executions
- Eloisa Sanchez/Reuters

**Notable Topics**

### Business
- China's AI upstart DeepSeek drops new model. Will it make waves like last year?
- Microsoft to offer voluntary retirement to thousands of US employees for the first time
- Consumer sentiment rebounds slightly after hitting lowest level on record
- Trump administration proposes eliminating all global reproductive health aid programs in new budget

### Health
- The HPV vaccine can reduce certain cancers by half. Expert explains why men and boys should get it
- Trump administration proposes eliminating all global reproductive health aid programs in new budget
- Cancer researchers see promising signs for mRNA vaccines after a year of turmoil

In [22]:
display_summary("https://anthropic.com")

# Summary of Anthropic Website

## Mission and Purpose
Anthropic is a public benefit corporation dedicated to securing the benefits and mitigating the risks of AI, with a focus on serving humanity's long-term well-being.

## Research and Development
- **Project Glasswing**: Securing critical software for the AI era.
- **Claude Opus 4.7**: A smarter and more capable Opus for coding, agents, vision, and complex professional work.
- **Anthropic Academy**: Building and learning with Claude.

## Products and Solutions
- **Claude**: A platform for building and deploying AI models.
- **Claude Code**: Code modernization and security solutions for AI development.
- **Claude Cowork**: Collaboration tools for AI teams.
- **Opus**, **Sonnet**, and **Haiku**: Various AI model solutions.

## Partnerships and Initiatives
- **Anthropic Code for Enterprise**: An enterprise version of Claude Code.
- **Cloud Partner Network**: Partnerships with major cloud providers (Amazon, Google Cloud, Microsoft).
- **Solutions**: AI agents, code modernization, customer support, education, financial services, healthcare, non-profits, security solutions.

## News and Announcements
### Recent Releases

* Claude Opus 4.7: A smarter, more capable Opus for coding, agents, vision, and complex professional work.
* Claude on Mars: The first AI-assisted drive on another planet (NASA's Perseverance rover).

### Blog Links: Available under 'Blog'

### News Articles:
- Anthropic’s Responsible Scaling Policy
- Alignment Science
- Anthropic’s Economic Index

In [ ]:
# Step 1: Create your prompts

system_prompt = "something here"
user_prompt = """
    Lots of text
    Can be pasted here
"""

# Step 2: Make the messages list

messages = [] # fill this in

# Step 3: Call OpenAI

response =

# Step 4: print the result

print(

## An extra exercise for those who enjoy web scraping

You may notice that if you try `display_summary("https://openai.com")` - it doesn't work! That's because OpenAI has a fancy website that uses Javascript. There are many ways around this that some of you might be familiar with. For example, Selenium is a hugely popular framework that runs a browser behind the scenes, renders the page, and allows you to query it. If you have experience with Selenium, Playwright or similar, then feel free to improve the Website class to use them. In the community-contributions folder, you'll find an example Selenium solution from a student (thank you!)